[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/salas/S2_repaso.ipynb)

# Repaso de S1 con MiBici

Repaso de unos 20 minutos al inicio de S2. No se entrega.
Corre las celdas en orden. Las líneas con un comentario `# cambien ...` son para que pruebes otros valores.

In [ ]:
# @title Preparación del entorno: solo córrela, no hace falta leerla
# Descarga los datos y dammg_utils.py, revisa que no lleguen dañados y avisa con ✅ entorno listo.
import hashlib, importlib.util, os, pathlib, subprocess, sys, urllib.request

BASE_URL = "https://hesusg.github.io/dammg-supervisado/datos/"
# Versiones fijas: copia generada de notebooks/requirements.txt (fuente única)
REQUISITOS = {
    'numpy': 'numpy==2.5.4',
    'pandas': 'pandas==3.0.6',
    'sklearn': 'scikit-learn==1.9.1',
    'matplotlib': 'matplotlib==3.11.2',
    'ipywidgets': 'ipywidgets==8.1.9',
    'pyarrow': 'pyarrow==26.0.0',
}
SHA256 = {
    'mibici_curso.parquet': 'a5fd844c57bbf220f0ddfc27dc965ef8fb8438bd6c1167709bde7ccc1dafa6d8',
    'dammg_utils.py': '7499b8695cb94d4dd72133e197d95a143e8b05b82f1d91928f6a8a30fa0f62d0',
}
EN_EVALUADOR = os.environ.get("DAMMG_GRADER") == "1"
# Umbral del evento |future_net| (percentil 85 de train), congelado para todo el curso
UMBRAL_EVENTO = 7.0


def _sha256(ruta):
    return hashlib.sha256(pathlib.Path(ruta).read_bytes()).hexdigest()


def _copia_local(nombre):
    """Busca una copia local: DAMMG_DATA_DIR o el repo del curso (data/curso, notebooks)."""
    carpetas = []
    if os.environ.get("DAMMG_DATA_DIR"):
        carpetas.append(pathlib.Path(os.environ["DAMMG_DATA_DIR"]))
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        carpetas += [p / "data" / "curso", p / "notebooks", p]
    for c in carpetas:
        if (c / nombre).is_file():
            return c / nombre
    return None


def _obtener(nombre, esperado):
    """Copia local si existe; si no, descarga del sitio del curso y verifica el sha256."""
    if EN_EVALUADOR:
        # Dentro del evaluador: nada de descargar; los datos ya están en /data
        return pathlib.Path("/data") / nombre
    local = _copia_local(nombre)
    if local is not None:
        if _sha256(local) != esperado:
            print(f"⚠ {nombre}: la copia local no coincide con el sha256 publicado (¿versión en desarrollo?)")
        return local
    destino = pathlib.Path("datos_dammg") / nombre
    destino.parent.mkdir(exist_ok=True)
    if not destino.exists() or _sha256(destino) != esperado:
        print(f"Descargando {nombre} ...")
        try:
            urllib.request.urlretrieve(BASE_URL + nombre, destino)
        except OSError as e:   # HTTPError, sin internet, etc.
            raise RuntimeError(
                f"No pude descargar {nombre} ({e}). Revisa tu conexión y vuelve a ejecutar esta celda. "
                "Si sigue fallando, avisa en el foro de Moodle: puede que el archivo aún no esté publicado."
            ) from None
    assert _sha256(destino) == esperado, f"{nombre} llegó incompleto o alterado: bórralo y vuelve a ejecutar."
    return destino


if not EN_EVALUADOR:
    faltan = [pin for modulo, pin in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
    if faltan:
        print("Instalando:", ", ".join(faltan))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
RUTAS = {nombre: _obtener(nombre, esperado) for nombre, esperado in SHA256.items()}

sys.path.insert(0, str(RUTAS["dammg_utils.py"].parent))
RUTA_DATOS = RUTAS["mibici_curso.parquet"]

import dammg_utils as du
du.entorno_listo()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
from dammg_utils import COLORES

du.estilo_dammg()

## 1. Una fila: una estación en una hora

In [ ]:
df = pd.read_parquet(RUTA_DATOS)
train = df[df.particion == "train"]   # octubre a junio: para ajustar
julio = df[df.particion == "val"]     # julio: para medir
print(f"train: {len(train):,} filas   julio: {len(julio):,} filas")

In [ ]:
estacion = 51   # cambien el número de estación
fila = julio[(julio.station_id == estacion) & (julio.hour == "2026-07-27 08:00")]
fila[["station_id", "hour", "departures_lag_168h", "departures"]]   # x = hace una semana, y = lo que salió

## 2. La regla de hace una semana

In [ ]:
x = fila.departures_lag_168h.iloc[0]   # lo que sabíamos antes
y = fila.departures.iloc[0]            # lo que pasó
y_gorro = x                            # la regla: lo mismo que hace una semana
print(f"x = {x:.0f}   ŷ = {y_gorro:.0f}   y = {y}   error = {y - y_gorro:.0f}")

## 3. El error de cada fila

In [ ]:
cinco = julio[(julio.station_id == 51) & julio.hour.between("2026-07-27 08:00", "2026-07-27 12:00")].copy()
cinco["error"] = (cinco.departures - cinco.departures_lag_168h).astype(int)   # y − ŷ
cinco[["hour_of_day", "departures_lag_168h", "departures", "error"]]

In [ ]:
print("suma:", cinco.error.sum(),                      # se cancelan
      "  absolutos:", cinco.error.abs().sum(),
      "  cuadrados:", (cinco.error ** 2).sum())

## 4. El costo J

In [ ]:
w = 0.7   # cambien el valor de w
y_gorro = w * train.departures_lag_168h + 1.18           # la recta con b fijo en 1.18
J = ((train.departures - y_gorro) ** 2).mean() / 2       # la mitad del promedio del error al cuadrado
print(f"w = {w}   J = {J:.2f}")

In [ ]:
costos = pd.DataFrame({"w": [i / 20 for i in range(6, 23)]})   # w de 0.30 a 1.10
costos["J"] = [((train.departures - v * train.departures_lag_168h - 1.18) ** 2).mean() / 2 for v in costos.w]
sns.lineplot(data=costos, x="w", y="J", marker="o", color=COLORES["w"]).set(title="Costo J según w")
plt.show()

## 5. La recta aprendida

In [ ]:
recta = LinearRegression().fit(train[["departures_lag_168h"]], train.departures)
print(f"w = {recta.coef_[0]:.2f}   b = {recta.intercept_:.2f}")

In [ ]:
prom = train.groupby("departures_lag_168h", as_index=False).departures.mean().query("departures_lag_168h <= 60")
prom["recta"] = recta.predict(prom[["departures_lag_168h"]])
ax = sns.scatterplot(data=prom, x="departures_lag_168h", y="departures", color="gray")
sns.lineplot(data=prom, x="departures_lag_168h", y="recta", color=COLORES["w"], ax=ax); plt.show()

## 6. Un dato raro

In [ ]:
dia = train[(train.station_id == 51) & train.hour.between("2026-06-29 00:00", "2026-06-29 23:00")]   # 24 filas
falsa = pd.DataFrame({"departures_lag_168h": [10], "departures": [300]})   # cambien 300 por 30
for nombre, datos in [("sin la fila falsa", dia), ("con la fila falsa", pd.concat([dia, falsa]))]:
    r = LinearRegression().fit(datos[["departures_lag_168h"]], datos.departures)
    print(f"{nombre}: w = {r.coef_[0]:.2f}   b = {r.intercept_:.2f}")

## 7. Train contra julio: MAE, RMSE y R²

In [ ]:
y_regla = julio.departures_lag_168h                       # la regla: lo de hace una semana
y_recta = recta.predict(julio[["departures_lag_168h"]])   # la recta aprendida en train

In [ ]:
def mide(pred):
    return {"MAE": mean_absolute_error(julio.departures, pred),
            "RMSE": root_mean_squared_error(julio.departures, pred), "R²": r2_score(julio.departures, pred)}
pd.DataFrame({"regla": mide(y_regla), "recta": mide(y_recta)}).T.round(2)

## 8. Polinomial: más curva no siempre es mejor

In [ ]:
semana = train[(train.station_id == 51) & train.hour.between("2026-06-15", "2026-06-21 23:00")]   # 168 filas
julio51 = julio[julio.station_id == 51]

In [ ]:
modelo = make_pipeline(MinMaxScaler((-1, 1)), PolynomialFeatures(degree=3),  # cambien el grado: 1, 3 u 8
                       LinearRegression()).fit(semana[["departures_lag_168h"]], semana.departures)
mae = lambda d: mean_absolute_error(d.departures, modelo.predict(d[["departures_lag_168h"]]))
print(f"MAE train: {mae(semana):.2f}   MAE julio: {mae(julio51):.2f}")

In [ ]:
malla = pd.DataFrame({"departures_lag_168h": range(0, 84)})   # de 0 a 83, lo que se ve en julio
malla["modelo"] = modelo.predict(malla)
ax = sns.scatterplot(data=semana, x="departures_lag_168h", y="departures", color="gray")
sns.lineplot(data=malla, x="departures_lag_168h", y="modelo", color=COLORES["w"], ax=ax).set(ylim=(0, 90)); plt.show()

## 9. Tres preguntas

Escriban la letra en la hoja, debajo de cada pregunta (doble clic para escribir).

**1. Dos modelos medidos en julio: A con MAE 2.3 y B con MAE 3.1. ¿Cuál falla por menos bicis en una hora normal?**

- a) A
- b) B
- c) No se puede saber

**Su respuesta (doble clic para escribir):** ___

**2. ¿Por qué no basta con sumar los errores?**

- a) Porque tarda mucho
- b) Porque los de arriba se cancelan con los de abajo
- c) Porque da negativo siempre

**Su respuesta (doble clic para escribir):** ___

**3. Un modelo mejora mucho en train y empeora en julio. ¿Qué le pasa?**

- a) Sobreajuste
- b) Le faltan datos de julio
- c) Es un outlier

**Su respuesta (doble clic para escribir):** ___